# Testy kalendarza

## 1. Podstawowe info
- shape, kolumny, typy danych
- zakres dat, liczba SKU

## 2. Kompletność kalendarza
- czy każde SKU ma ciągły zakres dat (bez luk)
- czy DataStart = pierwsza sprzedaż per SKU
- czy DataKoniec <= data_max_global

## 3. Dni zerowe
- ile rekordów z DokId=-1
- czy wypełnione wszystkie pola (brak NaN gdzie nie powinno być)
- czy IloscPlus=0, Wartosc=0 dla dni zerowych

## 4. Dni ze sprzedażą
- czy IloscPlus > 0 dla DokId != -1
- czy brak DokId=-1 wśród realnych transakcji

## 5. CenaPoRab
- czy zero NaN po ffill
- czy wartości sensowne (>0, brak outlierów)

## 6. Kolumny stałe per SKU
- czy NazwaTow, AsId, NazwaAsort są wypełnione dla wszystkich rekordów
- czy jeden SKU ma zawsze tę samą NazwaTow/AsId

## 7. Duplikaty
- czy są duplikaty całych rekordów
- czy są duplikaty [TowId, Data, DokId]

In [ ]:
import pandas as pd
import numpy as np

kalendarz_full = pd.read_parquet("dane/interim/kalendarz_full.parquet")

In [ ]:
# 1. Podstawowe info

rows, cols = kalendarz_full.shape
print(f"Rekordów: {rows:,}\nKolumn: {cols}")
print(f"\nKolumny:\n{kalendarz_full.columns.tolist()}")
print(f"\nTypy danych:\n{kalendarz_full.dtypes}")
print(f"\nUnikalnych SKU: {kalendarz_full['TowId'].nunique():,}")
print(f"Zakres dat: {kalendarz_full['Data'].min()} → {kalendarz_full['Data'].max()}")

In [ ]:
# 2. Kompletność kalendarza

# Czy każde SKU ma ciągły zakres dat (bez luk)
def sprawdz_luki(df):
    luki = (
        df.groupby('TowId')['Data']
        .apply(lambda x: x.sort_values().diff().dt.days.max())
    )
    return luki[luki > 1]

luki = sprawdz_luki(kalendarz_full)
print(f"SKU z lukami w kalendarzu: {len(luki)}")
if len(luki) > 0:
    print(luki.head(10))

# Czy DataStart = pierwsza sprzedaż per SKU
pierwsza_sprzedaz = (
    kalendarz_full[kalendarz_full['DokId'] != -1]
    .groupby('TowId')['Data'].min()
)
pierwsza_w_kalendarzu = kalendarz_full.groupby('TowId')['Data'].min()

roznice = (pierwsza_sprzedaz != pierwsza_w_kalendarzu).sum()
print(f"\nSKU gdzie DataStart != pierwsza sprzedaż: {roznice}")

# Czy DataKoniec <= data_max_global
data_max_global = kalendarz_full['Data'].max()
ostatni_dzien = kalendarz_full.groupby('TowId')['Data'].max()
print(f"\nSKU gdzie DataKoniec > data_max_global: {(ostatni_dzien > data_max_global).sum()}")

In [ ]:
# 3. Dni zerowe
dni_zerowe = kalendarz_full[kalendarz_full['DokId'] == -1]
dni_realne = kalendarz_full[kalendarz_full['DokId'] != -1]

print(f"Rekordów dni zerowych: {len(dni_zerowe):,}")
print(f"Rekordów dni realnych: {len(dni_realne):,}")

# Czy wypełnione wszystkie pola
print(f"\nNaN w dniach zerowych per kolumna:")
print(dni_zerowe.isna().sum()[dni_zerowe.isna().sum() > 0])

# Czy IloscPlus=0, Wartosc=0
print(f"\nIloscPlus != 0 w dniach zerowych: {(dni_zerowe['IloscPlus'] != 0).sum()}")
print(f"Wartosc != 0 w dniach zerowych: {(dni_zerowe['Wartosc'] != 0).sum()}")

In [ ]:
# 4. Dni ze sprzedażą
print(f"IloscPlus > 0 dla DokId != -1: {(dni_realne['IloscPlus'] > 0).sum():,}")
print(f"IloscPlus = 0 dla DokId != -1: {(dni_realne['IloscPlus'] == 0).sum():,}")
print(f"\nDokId = -1 wśród realnych transakcji: {(dni_realne['DokId'] == -1).sum()}")
print(f"\nTypDok w dniach realnych:\n{(dni_realne['TypDok'].value_counts())}")

In [ ]:
# 5. CenaPoRab
print(f"NaN w CenaPoRab: {kalendarz_full['CenaPoRab'].isna().sum():,}")
print(f"\nStatystyki CenaPoRab:")
print(kalendarz_full['CenaPoRab'].describe().round(2))
print(f"\nCenaPoRab <= 0: {(kalendarz_full['CenaPoRab'] <= 0).sum():,}")
print(f"CenaPoRab > 1000: {(kalendarz_full['CenaPoRab'] > 1000).sum():,}")

In [ ]:
# 6. Kolumny stałe per SKU
print("NaN w kolumnach stałych:")
for col in ['NazwaTow', 'AsId', 'NazwaAsort', 'EAN', 'Opis1']:
    nan_count = kalendarz_full[col].isna().sum()
    print(f"  {col}: {nan_count:,}")

# Czy jeden SKU ma zawsze tę samą NazwaTow i AsId
nazwy_per_sku = kalendarz_full.groupby('TowId')['NazwaTow'].nunique()
print(f"\nSKU z >1 NazwaTow: {(nazwy_per_sku > 1).sum()}")

asid_per_sku = kalendarz_full.groupby('TowId')['AsId'].nunique()
print(f"SKU z >1 AsId: {(asid_per_sku > 1).sum()}")

In [ ]:
# 8. Duplikaty
print(f"Duplikaty całych rekordów: {kalendarz_full.duplicated().sum():,}")

print(f"\nDuplikaty [TowId, Data, DokId]: {kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId']).sum():,}")

---

In [ ]:
mask_dup = kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId'], keep=False)
df_dup = kalendarz_full[mask_dup]

rozne_ceny = (
    df_dup.groupby(['TowId', 'Data', 'DokId'])['CenaPoRab']
    .nunique()
)
print(f"Grup z >1 ceną: {(rozne_ceny > 1).sum():,}")
print(f"Grup z tą samą ceną: {(rozne_ceny == 1).sum():,}")

In [ ]:
mask_dup = kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId'], keep=False)
kalendarz_full[mask_dup].sort_values(['DokId', 'TowId']).head(10)[
    ['DokId', 'TowId', 'Data', 'Kolejnosc', 'NrPozycji', 'IloscPlus', 'CenaPoRab']
]